In [ ]:
pip install kagglehub pandas

In [2]:
def print_section(title, width=65, color_code="\033[1;32m"):
    reset = "\033[0m"
    print("=" * width)
    print(f"{color_code}{title}{reset}")
    print("=" * width)

In [ ]:
import os
import pandas as pd
import kagglehub

# ETAPA 1 — LOCALIZAÇÃO DO DATASET ORIGINAL (sem re-download)
# O kagglehub mantém cache local em ~/.cache/kagglehub.
# force_download=False garante que o dataset já baixado não seja
# baixado novamente, economizando os 9,42 GB de tráfego.
print_section("ETAPA 1 — LOCALIZANDO DATASET NO CACHE LOCAL")

DATASET_ID = "gagansomashekar/microservices-bottleneck-detection-dataset"

DATASET_DIR = kagglehub.dataset_download(DATASET_ID, force_download=False)
print(f"✅ Dataset localizado em:\n   {DATASET_DIR}")

# Subpasta de interesse: processed_dataset contém os CSVs já
# processados com métricas de container, traces e labels de bottleneck.
# As pastas raw_dataset/ contêm arquivos intermediários menos limpos.
PASTA_PROCESSADA = os.path.join(DATASET_DIR, "processed_dataset")


# ETAPA 2 — MAPEAMENTO E CLASSIFICAÇÃO DOS ARQUIVOS
# A pasta processed_dataset contém 556 arquivos CSV divididos em três
# subpastas (compose, home, user) e dois grafos (graph_1, graph_2).
#
# Nomenclatura dos arquivos:
#   {tipo_bottleneck}_{data}_{duracao}_{rps}_{servico_afetado}_graph_{n}.csv
#
# Tipos de bottleneck identificados pelo prefixo:
#   cpu        → saturação de CPU
#   mem        → esgotamento de memória
#   net        → congestionamento de rede
#   cpu_mem    → CPU + memória simultâneos (prefixo "cpu_mem")
print_section("ETAPA 2 — MAPEANDO ARQUIVOS DA PASTA PROCESSADA")


def listar_csvs_processados(pasta):
    """Lista todos os CSVs de processed_dataset que contêm 'graph'."""
    resultado = []
    for raiz, _, nomes in os.walk(pasta):
        for nome in nomes:
            if nome.endswith(".csv") and "graph" in nome:
                resultado.append(
                    {
                        "caminho": os.path.join(raiz, nome),
                        "nome": nome,
                    }
                )
    return sorted(resultado, key=lambda x: x["nome"])


def classificar_tipo(nome_arquivo):
    """
    Classifica o tipo de bottleneck pelo prefixo do nome do arquivo.
    Ordem importante: 'cpu_mem' deve ser testado ANTES de 'cpu',
    pois cpu_mem também começa com 'cpu'.
    """
    n = nome_arquivo.lower()
    if n.startswith("cpu_mem"):
        return "cpu+mem"
    elif n.startswith("cpu"):
        return "cpu"
    elif n.startswith("mem"):
        return "mem"
    elif n.startswith("net"):
        return "net"
    else:
        return "outro"


arquivos = listar_csvs_processados(PASTA_PROCESSADA)

por_tipo = {"cpu": [], "mem": [], "net": [], "cpu+mem": []}
for a in arquivos:
    tipo = classificar_tipo(a["nome"])
    if tipo in por_tipo:
        a["tipo"] = tipo
        por_tipo[tipo].append(a)

print(f"Total de arquivos CSV encontrados: {len(arquivos)}")
for tipo, lista in por_tipo.items():
    print(f"  {tipo:<10} → {len(lista)} arquivos")


# ETAPA 3 — SELEÇÃO DOS ARQUIVOS MAIS REPRESENTATIVOS
# Estratégia de seleção: para cada tipo de bottleneck, selecionar os
# 3 arquivos com maior proporção de anomalias nas primeiras 2.000
# linhas (amostra rápida sem carregar o arquivo completo).
#
# Justificativa: arquivos com anomalias logo no início cobrem tanto
# o período de operação normal (linha de base) quanto o período de
# bottleneck ativo, garantindo representatividade de ambas as classes.
#
# Regra de desempate: quando dois arquivos têm a mesma pontuação
# (ex: ambos com 0% nas primeiras 2.000 linhas), a ordem alfabética
# do nome do arquivo determina a prioridade — comportamento padrão
# de sort() em Python com sort estável.
print_section("ETAPA 3 — SELECIONANDO 3 ARQUIVOS POR TIPO DE BOTTLENECK")

N_ARQUIVOS_POR_TIPO = 3  # top-3 por tipo
N_LINHAS_AMOSTRA = 2000  # linhas lidas para pontuar cada arquivo


def pontuar_arquivo(caminho):
    """
    Lê as primeiras N_LINHAS_AMOSTRA linhas e retorna a fração de
    registros com label_trace == 1 (anomalia/bottleneck ativo).
    Retorna 0.0 se o arquivo não tiver a coluna label_trace.
    """
    try:
        df = pd.read_csv(caminho, nrows=N_LINHAS_AMOSTRA)
        if "label_trace" not in df.columns:
            return 0.0
        return float(df["label_trace"].mean())
    except Exception:
        return 0.0


selecionados = []

for tipo, lista in por_tipo.items():
    pontuados = [(pontuar_arquivo(a["caminho"]), a) for a in lista]

    # Ordena por score descendente (sort estável mantém ordem alfabética
    # nos empates, pois a lista já estava ordenada por nome)
    pontuados.sort(key=lambda x: x[0], reverse=True)

    top = pontuados[:N_ARQUIVOS_POR_TIPO]

    print(f"\n  [{tipo}]")
    for score, a in top:
        print(f"    • {a['nome']:<55}  score={score*100:.1f}%")
        selecionados.append(a)


# ETAPA 4 — CARREGAMENTO COMPLETO E CONCATENAÇÃO
# Cada arquivo é lido por completo e recebe duas colunas extras:
#
#   tipo_bottleneck : str  — tipo do bottleneck injetado naquele experimento
#                            ("cpu", "mem", "net" ou "cpu+mem")
#   arquivo_origem  : str  — nome do arquivo-fonte (rastreabilidade)
#
# Os DataFrames são acumulados em lista e concatenados de uma só vez
# com pd.concat(ignore_index=True), evitando o PerformanceWarning
# causado por inserções coluna-a-coluna em DataFrames fragmentados.
print_section("ETAPA 4 — CARREGANDO ARQUIVOS E CONCATENANDO")

partes = []
for a in selecionados:
    df_parte = pd.read_csv(a["caminho"])
    # assign() é preferível a df["col"] = valor pois não fragmenta o DF
    df_parte = df_parte.assign(
        tipo_bottleneck=a["tipo"],
        arquivo_origem=a["nome"],
    )
    partes.append(df_parte)
    print(f"  ✔ {a['nome']}  ({len(df_parte):,} linhas)")

df_final = pd.concat(partes, ignore_index=True)
print(f"\nDataset final: {df_final.shape[0]:,} linhas × {df_final.shape[1]} colunas")


# ETAPA 5 — SALVAR O DATASET FINAL
ARQUIVO_SAIDA = "amostra_dataset_reduzida.csv"
df_final.to_csv(ARQUIVO_SAIDA, index=False)
tamanho_mb = os.path.getsize(ARQUIVO_SAIDA) / (1024 * 1024)
print(f"\n✅ Dataset salvo: '{ARQUIVO_SAIDA}'  ({tamanho_mb:.1f} MB)")